# Walk-Forward Optimization (WFO)

This notebook performs Walk-Forward Optimization using the `ggTrader` orchestrator api. It implements a **Robustness-First** selection strategy, picking parameters that perform consistently across multiple time folds rather than just the most recent one.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import vectorbt as vbt
import matplotlib.pyplot as plt
from tabulate import tabulate

# Auto-reload custom modules
%load_ext autoreload
%autoreload 2

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if project_root not in sys.path:
    sys.path.append(project_root)

from ggTrader.core.orchestrator import run_wfo_orchestrator

print("Environment initialized.")

In [ ]:
# --- Configuration ---
CONSTANTS = {
    "SYMBOLS": None,
    "SYMBOLS_FILE": os.path.join(os.getcwd(), "..", "..", "data", "top_20_USD_1095_movers.json"),
    "START_DATE": "2023-01-01",
    "END_DATE": "2025-12-31",
    "INTERVAL": "4h",
    "START_CASH": 10000,
    "PORTFOLIO_SHARE": 0.10,
    "FEES": 0.001,
    "N_SPLITS": 5,
    "TEST_RATIO": 0.334,
}

print("Configuration loaded.")

In [ ]:
# --- Define Parameter Grid ---
params = {
    "adx_threshold": [15, 25, 35],
    "adx_length": [14],
    "sar_acceleration": [0.02],
    "sar_maximum": [0.2],
    "atr_multiplier": [2.0, 3.0, 4.0],
    "atr_length": [14],
    "use_dmp_cross": [True, False],
}

print("Parameter grid defined.")

In [ ]:
# --- Run WFO ---
results = run_wfo_orchestrator(
    config=CONSTANTS, 
    param_grid=params, 
    save_results=False,
    show_progress=True
)

wfo_stats = results["wfo_stats"]
robust_top_5 = results["robust_top_5"]
final_pf = results["final_portfolio"]

print("\nWFO Complete.")

In [ ]:
# --- Analysis & Visualization ---

print("\nPARAMETER ROBUSTNESS REPORT (Top 5 Overall):")
robust_report_df = pd.DataFrame([{"Score": r["robustness_score"], **r["params"]} for r in robust_top_5])
print(tabulate(robust_report_df, headers="keys", tablefmt="github", showindex=False))

print("\nWFO FOLD RESULTS SUMMARY:")
df_results = pd.DataFrame(wfo_stats)
print(tabulate(df_results, headers="keys", tablefmt="github"))

print("\nFinal Model Performance (Full History with Best Robust Params):")
final_pf.plot().show()